# 3. Datenspeicherung in SQLite

## Ziel

Die aufbereiteten Daten werden in einer SQLite-Datenbank gespeichert und führen dann die SQL-Abfragen direkt aus Python aus.

Wir übernehmen:
- die BFS-Gemeindestammdaten,
- die bereinigten einzelnen Tarifwerte,
- die berechneten Tarifvergleichswerte je Gemeinde.

Die Tabellen werden über die Gemeindenummer miteinander verbunden.
Die detaillierten Tarifdaten bleiben neben den Gemeindewerten erhalten.

In [10]:
from pathlib import Path
import sqlite3
import pandas as pd

arbeitsordner = Path.cwd()

projektordner = (
    arbeitsordner.parent
    if arbeitsordner.name == "notebooks"
    else arbeitsordner
)

processed_ordner = projektordner / "data" / "processed"

print("Datenordner:", processed_ordner)
print("Ordner vorhanden:", processed_ordner.is_dir())

Datenordner: /Users/leyla.erikci/Desktop/data-analytics-group-05/data/processed
Ordner vorhanden: True


In [11]:
# Vollständige BFS-Gemeindestammdaten.
df_gemeinden = pd.read_csv(
    processed_ordner / "bfs_gemeinden_2026.csv",
    dtype={"gemeindenummer": "Int64"}
)

# Bereinigte einzelne Tarifwerte.
df_tarife = pd.read_csv(
    processed_ordner / "tarife_bereinigt.csv",
    dtype={"gemeindenummer_2026": "Int64"}
)

# Ein Tarifvergleichswert je Gemeinde.
df_gemeindewerte = pd.read_csv(
    processed_ordner / "gemeindewerte.csv",
    dtype={"gemeindenummer_2026": "Int64"}
)

print("Gemeindestammdaten:", len(df_gemeinden))
print("Einzelne Tarifwerte:", len(df_tarife))
print("Gemeindewerte:", len(df_gemeindewerte))

Gemeindestammdaten: 2110
Einzelne Tarifwerte: 2255
Gemeindewerte: 2108


## 3.1 Datenbanktabellen erstellen

Wir speichern die Daten hier: data/stromtarife.sqlite.

Die Tabelle "gemeinden" enthält die Gemeindestammdaten.
Die Tabelle "tarife" enthält die einzelnen bereinigten Tarifwerte.
Die Tabelle "gemeindewerte" enthält einen Tarifvergleichswert
je Gemeinde.

Gemeindename, Kanton und Gemeindetyp speichern wir zentral
in der Tabelle gemeinden. Über die Gemeindenummer können
diese Angaben mit den Tarif- und Gemeindewerten verknüpft werden.

In [12]:
# Gemeindestammdaten.
df_gemeinden_sql = df_gemeinden[
    [
        "gemeindenummer",
        "gemeindename",
        "kanton_id",
        "kanton",
        "gemeindetyp_code",
        "gemeindetyp"
    ]
].copy()

# Einzelne Tarife mit einheitlich benannter Gemeindenummer.
df_tarife_sql = (
    df_tarife.rename(
        columns={"gemeindenummer_2026": "gemeindenummer"}
    )[
        ["gemeindenummer", "betreiber", "gesamttarif"]
    ]
    .copy()
)

# Gemeindewerte ohne nochmals gespeicherte Stammdaten.
df_gemeindewerte_sql = (
    df_gemeindewerte.rename(
        columns={"gemeindenummer_2026": "gemeindenummer"}
    )[
        ["gemeindenummer", "tarif_gemeinde"]
    ]
    .copy()
)

# Schlüssel vor dem Speichern kontrollieren.
assert df_gemeinden_sql["gemeindenummer"].notna().all()
assert df_gemeinden_sql["gemeindenummer"].is_unique

assert df_gemeindewerte_sql["gemeindenummer"].notna().all()
assert df_gemeindewerte_sql["gemeindenummer"].is_unique

assert df_tarife_sql["gemeindenummer"].isin(
    df_gemeinden_sql["gemeindenummer"]
).all()

assert df_gemeindewerte_sql["gemeindenummer"].isin(
    df_gemeinden_sql["gemeindenummer"]
).all()

print("Tabellen für die Datenbank vorbereitet.")

Tabellen für die Datenbank vorbereitet.


In [13]:
from contextlib import closing

db_pfad = projektordner / "data" / "stromtarife.sqlite"

with closing(sqlite3.connect(db_pfad)) as verbindung:

    df_gemeinden_sql.to_sql(
        "gemeinden",
        verbindung,
        if_exists="replace",
        index=False
    )

    df_tarife_sql.to_sql(
        "tarife",
        verbindung,
        if_exists="replace",
        index=False
    )

    df_gemeindewerte_sql.to_sql(
        "gemeindewerte",
        verbindung,
        if_exists="replace",
        index=False
    )

print("Datenbank gespeichert:", db_pfad)

Datenbank gespeichert: /Users/leyla.erikci/Desktop/data-analytics-group-05/data/stromtarife.sqlite


In [14]:
sql = """
SELECT 'gemeinden' AS tabelle, COUNT(*) AS anzahl
FROM gemeinden

UNION ALL

SELECT 'tarife' AS tabelle, COUNT(*) AS anzahl
FROM tarife

UNION ALL

SELECT 'gemeindewerte' AS tabelle, COUNT(*) AS anzahl
FROM gemeindewerte;
"""

with closing(sqlite3.connect(db_pfad)) as verbindung:
    datenbank_bilanz = pd.read_sql_query(sql, verbindung)

display(datenbank_bilanz)

,tabelle,anzahl
0,gemeinden,2110
1,tarife,2255
2,gemeindewerte,2108


## 3.2 Tabellen mit SQL verknüpfen und zusammenfassen

Wir verbinden gemeindewerte und gemeinden über die Gemeindenummer.

Anschliessend berechnen wir je Gemeindetyp:
- die Anzahl Gemeinden,
- den durchschnittlichen Tarifvergleichswert,
- den niedrigsten und höchsten Tarifvergleichswert.

Wir verwenden die Gemeindewerte, damit jede Gemeinde einmal zählt.
Die Ergebnisse sind unbereinigt: Der Kanton wird hier noch nicht
statistisch berücksichtigt.

In [15]:
sql_join = """
SELECT
    g.gemeindenummer,
    g.gemeindename,
    g.kanton_id,
    g.kanton,
    g.gemeindetyp_code,
    g.gemeindetyp,
    w.tarif_gemeinde

FROM gemeindewerte AS w

INNER JOIN gemeinden AS g
    ON w.gemeindenummer = g.gemeindenummer

ORDER BY g.gemeindenummer;
"""

with closing(sqlite3.connect(db_pfad)) as verbindung:
    df_analyse_sql = pd.read_sql_query(
        sql_join,
        verbindung
    )

print("Gemeinden nach dem SQL-Join:", len(df_analyse_sql))

assert len(df_analyse_sql) == len(df_gemeindewerte_sql)
assert df_analyse_sql["gemeindenummer"].is_unique

display(df_analyse_sql.head())

Gemeinden nach dem SQL-Join: 2108


,gemeindenummer,gemeindename,kanton_id,kanton,gemeindetyp_code,gemeindetyp,tarif_gemeinde
0,1,Aeugst am Albis,1,Zürich,3,Ländlich,24.136
1,2,Affoltern am Albis,1,Zürich,1,Städtisch,24.136
2,3,Bonstetten,1,Zürich,2,Intermediär,24.136
3,4,Hausen am Albis,1,Zürich,2,Intermediär,24.136
4,5,Hedingen,1,Zürich,1,Städtisch,24.136


In [16]:
sql_gruppen = """
SELECT
    g.gemeindetyp_code,
    g.gemeindetyp,
    COUNT(*) AS anzahl_gemeinden,
    ROUND(AVG(w.tarif_gemeinde), 3) AS mittelwert_rp_kwh,
    ROUND(MIN(w.tarif_gemeinde), 3) AS minimum_rp_kwh,
    ROUND(MAX(w.tarif_gemeinde), 3) AS maximum_rp_kwh

FROM gemeindewerte AS w

INNER JOIN gemeinden AS g
    ON w.gemeindenummer = g.gemeindenummer

GROUP BY
    g.gemeindetyp_code,
    g.gemeindetyp

ORDER BY g.gemeindetyp_code;
"""

with closing(sqlite3.connect(db_pfad)) as verbindung:
    df_gruppen_sql = pd.read_sql_query(
        sql_gruppen,
        verbindung
    )

assert df_gruppen_sql["anzahl_gemeinden"].sum() == len(
    df_analyse_sql
)

display(df_gruppen_sql)

,gemeindetyp_code,gemeindetyp,anzahl_gemeinden,mittelwert_rp_kwh,minimum_rp_kwh,maximum_rp_kwh
0,1,Städtisch,508,28.005,21.883,36.910
1,2,Intermediär,579,28.182,16.227,41.365
2,3,Ländlich,1021,28.361,9.643,43.613


## 3.3 Gemeinden ohne Tarifwert mit SQL ermitteln

Wir prüfen, welche Gemeinden aus der vollständigen BFS-Tabelle
keinen Eintrag in der Tabelle gemeindewerte besitzen.

Dazu verwenden wir:
- einen LEFT JOIN, damit alle BFS-Gemeinden erhalten bleiben,
- die Bedingung IS NULL, um fehlende Zuordnungen auszuwählen.

Wir erwarten Augst und Nürensdorf. Damit kontrollieren wir,
ob die Datenbank dasselbe Ergebnis liefert wie die vorherige
Datenaufbereitung.

Zusätzlich prüfen wir, ob Gemeinden mit und ohne Tarifwert
zusammen den vollständigen BFS-Bestand ergeben.

In [17]:
sql_ohne_tarif = """
SELECT
    g.gemeindenummer,
    g.gemeindename,
    g.kanton,
    g.gemeindetyp

FROM gemeinden AS g

LEFT JOIN gemeindewerte AS w
    ON g.gemeindenummer = w.gemeindenummer

WHERE w.gemeindenummer IS NULL

ORDER BY g.gemeindenummer;
"""

with closing(sqlite3.connect(db_pfad)) as verbindung:
    df_ohne_tarif_sql = pd.read_sql_query(
        sql_ohne_tarif,
        verbindung
    )

display(df_ohne_tarif_sql)

assert len(df_analyse_sql) + len(df_ohne_tarif_sql) == len(
    df_gemeinden_sql
)

,gemeindenummer,gemeindename,kanton,gemeindetyp
0,64,Nürensdorf,Zürich,Intermediär
1,2822,Augst,Basel-Landschaft,Städtisch


## 3.4 SQL-Abfragen und Ergebnisse speichern

Wir speichern die SQL-Abfragen als .sql-Dateien im Ordner sql.

Die Tabellenbilanz, die Zusammenfassung nach Gemeindetyp und
die Gemeinden ohne Tarifwert speichern wir unter results/sql
als CSV-Dateien.

So sind die Abfragen und Ergebnisse auch ausserhalb
des Notebooks nachvollziehbar.

In [18]:
sql_ordner = projektordner / "sql"
ergebnis_ordner = projektordner / "results" / "sql"

sql_ordner.mkdir(parents=True, exist_ok=True)
ergebnis_ordner.mkdir(parents=True, exist_ok=True)

# SQL-Abfragen speichern.
abfragen = {
    "01_tabellenbilanz.sql": sql,
    "02_gemeindewerte_join.sql": sql_join,
    "03_tarife_nach_gemeindetyp.sql": sql_gruppen,
    "04_gemeinden_ohne_tarif.sql": sql_ohne_tarif
}

for dateiname, abfrage in abfragen.items():
    (sql_ordner / dateiname).write_text(
        abfrage,
        encoding="utf-8"
    )

# Ergebnisse speichern.
datenbank_bilanz.to_csv(
    ergebnis_ordner / "tabellenbilanz.csv",
    index=False
)

df_gruppen_sql.to_csv(
    ergebnis_ordner / "tarife_nach_gemeindetyp.csv",
    index=False
)

df_ohne_tarif_sql.to_csv(
    ergebnis_ordner / "gemeinden_ohne_tarif.csv",
    index=False
)

print("SQL-Abfragen und Ergebnisse gespeichert.")

SQL-Abfragen und Ergebnisse gespeichert.
